# Formalización de la partición experimental

Este notebook formaliza la organización experimental del conjunto MMOTU / OTU_2D en `train`, `validation` y `test`.

La partición no se genera ni se remuestrea en este notebook. La membresía de cada subconjunto queda congelada tal como estaba registrada en el manifiesto anterior (`configs/splits/legacy/mmotu_experimental_split_legacy.csv`).

In [1]:
# Importación de librerias
import sys
import filecmp
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("../../").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.pipeline.pipeline import extract_technical_info
from src.pipeline.metadata import serialize_mask_values

## 1. Definición de rutas

- `LEGACY_MANIFEST_PATH`: manifiesto anterior, conservado sin modificaciones.
- `STANDARD_MANIFEST_PATH`: manifiesto de compatibilidad utilizado por los experimentos R4/R5. Solo se lee.
- `CANONICAL_MANIFEST_PATH`: manifiesto canónico de partición generado en este notebook.
- `TECHNICAL_METADATA_PATH`: metadatos técnicos recalculados desde la copia experimental.

In [2]:
SPLITS_DIR = PROJECT_ROOT / "configs" / "splits"

LEGACY_MANIFEST_PATH = (
    SPLITS_DIR
    / "legacy"
    / "mmotu_experimental_split_legacy.csv"
)

STANDARD_MANIFEST_PATH = (
    SPLITS_DIR
    / "mmotu_experimental_split_standard.csv"
)

CANONICAL_MANIFEST_PATH = (
    SPLITS_DIR
    / "mmotu_experimental_split.csv"
)

EXPERIMENTAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "MMOTU_OTU_2D_experimental"
)

TECHNICAL_METADATA_PATH = (
    PROJECT_ROOT
    / "results"
    / "dataset"
    / "mmotu_experimental_technical_metadata.csv"
)

EXPERIMENTAL_SPLITS = ["train", "validation", "test"]

for path in [
    LEGACY_MANIFEST_PATH,
    STANDARD_MANIFEST_PATH,
    CANONICAL_MANIFEST_PATH,
    EXPERIMENTAL_DIR,
    TECHNICAL_METADATA_PATH,
]:
    print(path.relative_to(PROJECT_ROOT).as_posix())

configs/splits/legacy/mmotu_experimental_split_legacy.csv
configs/splits/mmotu_experimental_split_standard.csv
configs/splits/mmotu_experimental_split.csv
data/processed/MMOTU_OTU_2D_experimental
results/dataset/mmotu_experimental_technical_metadata.csv


## 2. Carga del manifiesto anterior

El manifiesto anterior contiene, para cada muestra:

- identificador de imagen,
- subconjunto de procedencia en MMOTU (`source_split`),
- categoría diagnóstica original,
- etiqueta y clase binaria,
- partición experimental asignada con la nomenclatura anterior.

In [3]:
legacy_df = pd.read_csv(LEGACY_MANIFEST_PATH)

legacy_df.head()

,image_id,image_name,source_split,original_class,binary_label,binary_class,experimental_split
0,789,789.JPG,train,5,0,benign,train_internal
1,312,312.JPG,train,4,0,benign,train_internal
2,1196,1196.JPG,train,6,1,malignant,train_internal
3,297,297.JPG,train,1,0,benign,train_internal
4,263,263.JPG,train,2,0,benign,train_internal


In [4]:
legacy_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1469 entries, 0 to 1468
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   image_id            1469 non-null   int64 
 1   image_name          1469 non-null   object
 2   source_split        1469 non-null   object
 3   original_class      1469 non-null   int64 
 4   binary_label        1469 non-null   int64 
 5   binary_class        1469 non-null   object
 6   experimental_split  1469 non-null   object
dtypes: int64(3), object(4)
memory usage: 80.5+ KB


## 3. Distribución de muestras en el manifiesto anterior

In [5]:
legacy_df["experimental_split"].value_counts()

experimental_split
train_internal      800
final_validation    469
val_internal        200
Name: count, dtype: int64

In [6]:
pd.crosstab(legacy_df["experimental_split"], legacy_df["binary_class"])

binary_class,benign,malignant
experimental_split,,
final_validation,421,48
train_internal,713,87
val_internal,178,22


## 4. Construcción del manifiesto canónico

Se renombran las particiones sin alterar la membresía. Se conservan dos conceptos diferenciados:

- `original_split`: procedencia de la muestra dentro de MMOTU (`train` = 1000 imágenes, `validation` = 469 imágenes del subconjunto `val` original).
- `experimental_split`: uso de la muestra dentro de la tesis (`train`, `validation`, `test`).

Las rutas registradas corresponden a la copia experimental disponible (`data/processed/MMOTU_OTU_2D_experimental/`), relativas a la raíz del proyecto.

In [7]:
SPLIT_MAPPING = {
    "train_internal": "train",
    "val_internal": "validation",
    "final_validation": "test",
}

# Toda etiqueta anterior debe tener una correspondencia definida
assert set(legacy_df["experimental_split"]) == set(SPLIT_MAPPING)

canonical_df = pd.DataFrame({
    "image_id": legacy_df["image_id"],
    "image_name": legacy_df["image_name"],
    "mask_name": legacy_df["image_name"].map(lambda name: Path(name).stem + "_binary.PNG"),
    "original_split": legacy_df["source_split"],
    "original_class": legacy_df["original_class"],
    "binary_label": legacy_df["binary_label"],
    "binary_class": legacy_df["binary_class"],
    "experimental_split": legacy_df["experimental_split"].map(SPLIT_MAPPING),
    "legacy_split": legacy_df["experimental_split"],
})

canonical_df["experimental_image_path"] = [
    (EXPERIMENTAL_DIR / split / "images" / name).relative_to(PROJECT_ROOT).as_posix()
    for split, name in zip(canonical_df["experimental_split"], canonical_df["image_name"])
]

canonical_df["experimental_mask_path"] = [
    (EXPERIMENTAL_DIR / split / "masks" / name).relative_to(PROJECT_ROOT).as_posix()
    for split, name in zip(canonical_df["experimental_split"], canonical_df["mask_name"])
]

canonical_df.head()

,image_id,image_name,mask_name,original_split,original_class,binary_label,binary_class,experimental_split,legacy_split,experimental_image_path,experimental_mask_path
0,789,789.JPG,789_binary.PNG,train,5,0,benign,train,train_internal,data/processed/MMOTU_OTU_2D_experimental/train...,data/processed/MMOTU_OTU_2D_experimental/train...
1,312,312.JPG,312_binary.PNG,train,4,0,benign,train,train_internal,data/processed/MMOTU_OTU_2D_experimental/train...,data/processed/MMOTU_OTU_2D_experimental/train...
2,1196,1196.JPG,1196_binary.PNG,train,6,1,malignant,train,train_internal,data/processed/MMOTU_OTU_2D_experimental/train...,data/processed/MMOTU_OTU_2D_experimental/train...
3,297,297.JPG,297_binary.PNG,train,1,0,benign,train,train_internal,data/processed/MMOTU_OTU_2D_experimental/train...,data/processed/MMOTU_OTU_2D_experimental/train...
4,263,263.JPG,263_binary.PNG,train,2,0,benign,train,train_internal,data/processed/MMOTU_OTU_2D_experimental/train...,data/processed/MMOTU_OTU_2D_experimental/train...


In [8]:
canonical_df.to_csv(CANONICAL_MANIFEST_PATH, index=False)

print("Manifiesto canónico generado:", CANONICAL_MANIFEST_PATH.relative_to(PROJECT_ROOT))

Manifiesto canónico generado: configs/splits/mmotu_experimental_split.csv


## 5. Metadatos técnicos recalculados desde la copia experimental

Las propiedades técnicas se recalculan a partir de los archivos disponibles en `data/processed/MMOTU_OTU_2D_experimental/`, utilizando la misma función del pipeline (`extract_technical_info`).

**Este archivo no es el `metadata.csv` original generado desde `data/raw`.** Esa fuente no se encuentra disponible localmente, por lo que no se registran ni se infieren rutas hacia el dataset raw. Solo se incluyen propiedades intrínsecas verificadas sobre los archivos actuales.

In [9]:
technical_records = []

for _, row in canonical_df.iterrows():
    image_path = PROJECT_ROOT / row["experimental_image_path"]
    mask_path = PROJECT_ROOT / row["experimental_mask_path"]

    technical_info = extract_technical_info(
        pd.Series({"image_path": image_path, "mask_path": mask_path})
    )
    technical_info["mask_values"] = serialize_mask_values(technical_info["mask_values"])

    technical_records.append({
        "image_id": row["image_id"],
        "experimental_split": row["experimental_split"],
        "experimental_image_path": row["experimental_image_path"],
        "experimental_mask_path": row["experimental_mask_path"],
        **technical_info,
        "metadata_source": "recalculated_from_experimental_copy",
    })

technical_df = pd.DataFrame(technical_records)

technical_df.head()

,image_id,experimental_split,experimental_image_path,experimental_mask_path,image_width,image_height,image_channels,image_dtype,image_format,mask_width,mask_height,mask_channels,mask_dtype,mask_format,mask_values,same_dimensions,metadata_source
0,789,train,data/processed/MMOTU_OTU_2D_experimental/train...,data/processed/MMOTU_OTU_2D_experimental/train...,860,622,3,uint8,.JPG,860,622,1,uint8,.PNG,"0,1",True,recalculated_from_experimental_copy
1,312,train,data/processed/MMOTU_OTU_2D_experimental/train...,data/processed/MMOTU_OTU_2D_experimental/train...,479,532,3,uint8,.JPG,479,532,1,uint8,.PNG,"0,1",True,recalculated_from_experimental_copy
2,1196,train,data/processed/MMOTU_OTU_2D_experimental/train...,data/processed/MMOTU_OTU_2D_experimental/train...,958,534,3,uint8,.JPG,958,534,1,uint8,.PNG,"0,1",True,recalculated_from_experimental_copy
3,297,train,data/processed/MMOTU_OTU_2D_experimental/train...,data/processed/MMOTU_OTU_2D_experimental/train...,958,531,3,uint8,.JPG,958,531,1,uint8,.PNG,"0,1",True,recalculated_from_experimental_copy
4,263,train,data/processed/MMOTU_OTU_2D_experimental/train...,data/processed/MMOTU_OTU_2D_experimental/train...,480,532,3,uint8,.JPG,480,532,1,uint8,.PNG,"0,1",True,recalculated_from_experimental_copy


In [10]:
TECHNICAL_METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)

technical_df.to_csv(TECHNICAL_METADATA_PATH, index=False)

print("Metadatos técnicos generados:", TECHNICAL_METADATA_PATH.relative_to(PROJECT_ROOT))

Metadatos técnicos generados: results/dataset/mmotu_experimental_technical_metadata.csv


## 6. Validación de la partición experimental

Las comprobaciones se realizan sobre el manifiesto canónico leído nuevamente desde disco.

In [11]:
df = pd.read_csv(CANONICAL_MANIFEST_PATH)
technical_df = pd.read_csv(TECHNICAL_METADATA_PATH)

validation_results = []

def register(check: str, result) -> None:
    validation_results.append({"verificacion": check, "resultado": result})

### 6.1 Total de registros e identificadores únicos

In [12]:
assert len(df) == 1469
assert df["image_id"].nunique() == 1469

register("Total de registros", len(df))
register("image_id únicos", df["image_id"].nunique())

print("Registros:", len(df))
print("image_id únicos:", df["image_id"].nunique())

Registros: 1469
image_id únicos: 1469


### 6.2 Pertenencia a un único split y ausencia de intersección

In [13]:
assert set(df["experimental_split"]) == set(EXPERIMENTAL_SPLITS)
assert df["experimental_split"].notna().all()

splits_per_id = df.groupby("image_id")["experimental_split"].nunique()
assert (splits_per_id == 1).all()

split_ids = {
    split: set(df.loc[df["experimental_split"] == split, "image_id"])
    for split in EXPERIMENTAL_SPLITS
}

for i, split_a in enumerate(EXPERIMENTAL_SPLITS):
    for split_b in EXPERIMENTAL_SPLITS[i + 1:]:
        intersection = split_ids[split_a] & split_ids[split_b]
        print(f"{split_a} ∩ {split_b}:", len(intersection))
        assert len(intersection) == 0

register("image_id asignados a más de un split", int((splits_per_id != 1).sum()))
register("Intersección de image_id entre splits", 0)

train ∩ validation: 0
train ∩ test: 0
validation ∩ test: 0


### 6.3 Número de muestras por split y procedencia en MMOTU

In [14]:
split_counts = df["experimental_split"].value_counts().reindex(EXPERIMENTAL_SPLITS)

assert split_counts.to_dict() == {"train": 800, "validation": 200, "test": 469}

for split, count in split_counts.items():
    register(f"Muestras en {split}", int(count))

split_counts

experimental_split
train         800
validation    200
test          469
Name: count, dtype: int64

In [15]:
pd.crosstab(df["experimental_split"], df["original_split"]).reindex(EXPERIMENTAL_SPLITS)

original_split,train,validation
experimental_split,,
train,800,0
validation,200,0
test,0,469


In [16]:
original_train_ids = set(df.loc[df["original_split"] == "train", "image_id"])
original_validation_ids = set(df.loc[df["original_split"] == "validation", "image_id"])

# test corresponde exactamente al subconjunto val original de MMOTU
assert split_ids["test"] == original_validation_ids
# train + validation corresponden exactamente al subconjunto train original de MMOTU
assert (split_ids["train"] | split_ids["validation"]) == original_train_ids
assert len(original_train_ids) == 1000

register("test = val original de MMOTU (469)", True)
register("train ∪ validation = train original de MMOTU (1000)", True)

### 6.4 Distribución benigno / maligno por split

La proporción de casos malignos de cada split se compara con la del conjunto de desarrollo (1000 imágenes del subconjunto `train` original) y con la del conjunto completo. Esta comparación es descriptiva: no se declara el método con que se obtuvo la división 800/200.

In [17]:
class_distribution = (
    pd.crosstab(df["experimental_split"], df["binary_class"])
    .reindex(EXPERIMENTAL_SPLITS)
)

class_distribution["total"] = class_distribution.sum(axis=1)
class_distribution["malignant_pct"] = (
    100 * class_distribution["malignant"] / class_distribution["total"]
).round(2)

development_df = df[df["original_split"] == "train"]

reference = pd.DataFrame({
    "benign": [
        (development_df["binary_class"] == "benign").sum(),
        (df["binary_class"] == "benign").sum(),
    ],
    "malignant": [
        (development_df["binary_class"] == "malignant").sum(),
        (df["binary_class"] == "malignant").sum(),
    ],
}, index=["desarrollo (train original)", "conjunto completo"])

reference["total"] = reference.sum(axis=1)
reference["malignant_pct"] = (100 * reference["malignant"] / reference["total"]).round(2)

pd.concat([class_distribution, reference])

,benign,malignant,total,malignant_pct
train,713,87,800,10.88
validation,178,22,200,11.00
test,421,48,469,10.23
desarrollo (train original),891,109,1000,10.90
conjunto completo,1312,157,1469,10.69


### 6.5 Rutas faltantes y archivos no registrados

In [18]:
missing_images = [
    path for path in df["experimental_image_path"]
    if not (PROJECT_ROOT / path).exists()
]
missing_masks = [
    path for path in df["experimental_mask_path"]
    if not (PROJECT_ROOT / path).exists()
]

print("Imágenes faltantes:", len(missing_images))
print("Máscaras faltantes:", len(missing_masks))

assert len(missing_images) == 0
assert len(missing_masks) == 0

register("Rutas de imagen faltantes", len(missing_images))
register("Rutas de máscara faltantes", len(missing_masks))

Imágenes faltantes: 0
Máscaras faltantes: 0


In [19]:
orphan_files = 0

for split in EXPERIMENTAL_SPLITS:
    split_df = df[df["experimental_split"] == split]

    actual_images = {path.name for path in (EXPERIMENTAL_DIR / split / "images").iterdir() if path.is_file()}
    actual_masks = {path.name for path in (EXPERIMENTAL_DIR / split / "masks").iterdir() if path.is_file()}

    orphan_images = actual_images - set(split_df["image_name"])
    orphan_masks = actual_masks - set(split_df["mask_name"])

    print(split)
    print("Images:", len(actual_images), "| huérfanas:", len(orphan_images))
    print("Masks :", len(actual_masks), "| huérfanas:", len(orphan_masks))
    print("-" * 30)

    assert len(actual_images) == len(split_df)
    assert len(actual_masks) == len(split_df)

    orphan_files += len(orphan_images) + len(orphan_masks)

assert orphan_files == 0

register("Archivos en carpetas no registrados en el manifiesto", orphan_files)

train
Images: 800 | huérfanas: 0
Masks : 800 | huérfanas: 0
------------------------------
validation
Images: 200 | huérfanas: 0
Masks : 200 | huérfanas: 0
------------------------------
test
Images: 469 | huérfanas: 0
Masks : 469 | huérfanas: 0
------------------------------


### 6.6 Correspondencia imagen - máscara

In [20]:
# Mismo identificador en imagen y máscara
mask_ids = df["mask_name"].str.replace("_binary.PNG", "", regex=False)
assert (mask_ids == df["image_name"].map(lambda name: Path(name).stem)).all()

# Imagen y máscara en la misma partición
assert (df["experimental_image_path"].str.split("/").str[3] == df["experimental_split"]).all()
assert (df["experimental_mask_path"].str.split("/").str[3] == df["experimental_split"]).all()

# Mismas dimensiones y máscara binaria
assert technical_df["image_id"].tolist() == df["image_id"].tolist()
assert technical_df["same_dimensions"].all()
assert (technical_df["mask_channels"] == 1).all()

print("Pares con mismas dimensiones:", int(technical_df["same_dimensions"].sum()), "/", len(technical_df))
print("Valores de máscara:", technical_df["mask_values"].value_counts().to_dict())

register("Pares imagen–máscara con mismo id, split y dimensiones", f"{int(technical_df['same_dimensions'].sum())}/{len(technical_df)}")

Pares con mismas dimensiones: 1469 / 1469
Valores de máscara: {'0,1': 1469}


### 6.7 Imágenes duplicadas exactas

Se consideran duplicados exactos los archivos de imagen con contenido idéntico byte a byte. Solo se comparan entre sí los archivos que tienen el mismo tamaño.

In [21]:
images_df = df[["image_id", "experimental_split", "experimental_image_path"]].copy()
images_df["file_size"] = [
    (PROJECT_ROOT / path).stat().st_size for path in images_df["experimental_image_path"]
]

duplicate_pairs = []

for _, group in images_df.groupby("file_size"):
    rows = group.to_dict("records")

    for i, row_a in enumerate(rows):
        for row_b in rows[i + 1:]:
            if filecmp.cmp(
                PROJECT_ROOT / row_a["experimental_image_path"],
                PROJECT_ROOT / row_b["experimental_image_path"],
                shallow=False,
            ):
                duplicate_pairs.append((row_a, row_b))

cross_split_pairs = [
    (row_a, row_b) for row_a, row_b in duplicate_pairs
    if row_a["experimental_split"] != row_b["experimental_split"]
]

print("Duplicados exactos identificados:", len(duplicate_pairs))
print("Duplicados exactos entre particiones:", len(cross_split_pairs))

assert len(duplicate_pairs) == 0
assert len(cross_split_pairs) == 0

register("Duplicados exactos identificados", len(duplicate_pairs))
register("Duplicados exactos entre particiones", len(cross_split_pairs))

Duplicados exactos identificados: 0
Duplicados exactos entre particiones: 0


### 6.8 Equivalencia con los manifiestos anteriores

- Con `mmotu_experimental_split_standard.csv`: mismos `image_id` por split y mismas etiquetas. Este archivo se mantiene sin cambios por compatibilidad con los experimentos R4/R5.
- Con el manifiesto anterior (`legacy`): misma membresía tras renombrar las particiones.

In [22]:
standard_df = pd.read_csv(STANDARD_MANIFEST_PATH)

comparison = df.merge(standard_df, on="image_id", how="outer", suffixes=("", "_standard"), indicator=True)

assert (comparison["_merge"] == "both").all()
assert (comparison["experimental_split"] == comparison["experimental_split_standard"]).all()
assert (comparison["original_split"] == comparison["source_split"]).all()

for column in ["image_name", "original_class", "binary_label", "binary_class"]:
    assert (comparison[column] == comparison[f"{column}_standard"]).all()

legacy_comparison = df.merge(legacy_df, on="image_id", suffixes=("", "_legacy"))

assert len(legacy_comparison) == 1469
assert (legacy_comparison["legacy_split"] == legacy_comparison["experimental_split_legacy"]).all()
assert (legacy_comparison["experimental_split_legacy"].map(SPLIT_MAPPING) == legacy_comparison["experimental_split"]).all()

register("Membresía idéntica a mmotu_experimental_split_standard.csv", True)
register("Membresía idéntica al manifiesto anterior (legacy)", True)

print("Membresía y etiquetas equivalentes a _standard.csv y al manifiesto anterior.")

Membresía y etiquetas equivalentes a _standard.csv y al manifiesto anterior.


## 7. Resumen de verificaciones

In [23]:
pd.DataFrame(validation_results)

,verificacion,resultado
0,Total de registros,1469
1,image_id únicos,1469
2,image_id asignados a más de un split,0
3,Intersección de image_id entre splits,0
4,Muestras en train,800
5,Muestras en validation,200
6,Muestras en test,469
7,test = val original de MMOTU (469),True
8,train ∪ validation = train original de MMOTU (...,True
9,Rutas de imagen faltantes,0


## 8. Distribución final de la partición experimental

Conteos calculados a partir del manifiesto canónico `configs/splits/mmotu_experimental_split.csv`.

In [24]:
final_table = (
    pd.crosstab(df["experimental_split"], df["binary_class"])
    .reindex(EXPERIMENTAL_SPLITS)
    .rename(columns={"benign": "benignas", "malignant": "malignas"})
)

final_table["total"] = final_table.sum(axis=1)
final_table.loc["total"] = final_table.sum()
final_table.index.name = "split"
final_table.columns.name = None

final_table

,benignas,malignas,total
split,,,
train,713,87,800
validation,178,22,200
test,421,48,469
total,1312,157,1469
